<a href="https://colab.research.google.com/github/kelvin17-glitch/emotion_recognition/blob/master/notebooks/four.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="color: red; font-size: 40px; text-align: center;">TRANSFORMERS</div>

<div style="color: green; font-size: 30px;">1. Using Raw Audio</div>

In [ ]:
# Import relevant libraries
import os
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, accuracy_score

import torch
import torchaudio

In [ ]:
# Load data from drive
from google.colab import drive
drive.mount('/content/drive')

# Process data. End up with a list of (path, label), convert to df
EMOTION_MAP = {
    "ANG": "angry", "NEU": "neutral", "FEA": "fearful",
    "SAD": "sad", "HAP": "happy", "DIS": "disgust"
}

data = []
audio_dir = "/content/drive/MyDrive/data"
# Extract data
for filename in os.listdir(audio_dir):
    if filename.endswith('.wav'):
        emotion = filename.split("_")[2]
        emotion = EMOTION_MAP[emotion]
        path = os.path.join(audio_dir, filename)
        data.append((path, emotion))

df = pd.DataFrame(data, columns=["path", "label"])

Mounted at /content/drive


In [ ]:
# Load processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

# Convert pandas to HF dataset
dataset = Dataset.from_pandas(df)

# Preprocessing function
def preprocess(example):
    waveform, sample_rate = torchaudio.load(example["path"])
    waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
    example["input_values"] = processor(waveform.squeeze().numpy(), sampling_rate=16000).input_values[0]
    example["label"] = label2id[example["label"]]
    return example

label2id = {label: i for i, label in enumerate(df["label"].unique())}
id2label = {i: label for label, i in label2id.items()}

dataset = dataset.map(preprocess, remove_columns=["path"])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

Map:   0%|          | 0/7442 [00:00<?, ? examples/s]

In [ ]:
# Load model
model = Wav2Vec2ForSequenceClassification.from_pretrained("facebook/wav2vec2-base", num_labels=6, label2id=label2id, id2label=id2label)

# Split dataset
dataset_dict = dataset.train_test_split(test_size=0.2)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    num_train_epochs=10,
    save_strategy='epoch',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='Accuracy',
)

# Metrics
def compute_metrics(val_pred):
    logits, labels = val_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "Accuracy": accuracy_score(labels, preds)
    }

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics
)
trainer.train()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-6-c49a187668fb>:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mwndwa (mwndwa-moi-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,1.793400,1.801870,0.167898


Epoch,Training Loss,Validation Loss,Accuracy
1,1.793400,1.801870,0.167898
2,1.793200,1.791125,0.172599
3,1.791000,1.790032,0.172599
4,1.793100,1.790032,0.167898
5,1.789200,1.789873,0.174614
6,1.791500,1.789913,0.169913
7,1.788100,1.789843,0.169913
8,1.790100,1.789718,0.174614
9,1.789900,1.789778,0.167898
10,1.791300,1.789715,0.171927


TrainOutput(global_step=7450, training_loss=1.791708661789862, metrics={'train_runtime': 5801.9435, 'train_samples_per_second': 10.26, 'train_steps_per_second': 1.284, 'total_flos': 1.8083370658673009e+18, 'train_loss': 1.791708661789862, 'epoch': 10.0})

In [ ]:
# prompt: evaluate the transformer above

import numpy as np
# Evaluate the transformer model
results = trainer.evaluate(dataset_dict["test"])

# Print evaluation results
print("Evaluation Results:")
print(results)

# Generate predictions on the test set
predictions = trainer.predict(dataset_dict["test"])

# Get the predicted labels and true labels
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Convert label IDs back to label names for the report
predicted_label_names = [id2label[label_id] for label_id in predicted_labels]
true_label_names = [id2label[label_id] for label_id in true_labels]

# Print classification report
print("\nClassification Report:")
print(classification_report(true_label_names, predicted_label_names, target_names=list(id2label.values())))

Evaluation Results:
{'eval_loss': 1.7898725271224976, 'eval_Accuracy': 0.17461383478844864, 'eval_runtime': 49.5148, 'eval_samples_per_second': 30.072, 'eval_steps_per_second': 3.777, 'epoch': 10.0}

Classification Report:
              precision    recall  f1-score   support

       happy       0.00      0.00      0.00       257
         sad       0.00      0.00      0.00       253
       angry       0.00      0.00      0.00       250
     neutral       0.00      0.00      0.00       256
     disgust       0.00      0.00      0.00       213
     fearful       0.17      1.00      0.30       260

    accuracy                           0.17      1489
   macro avg       0.03      0.17      0.05      1489
weighted avg       0.03      0.17      0.05      1489



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
